In [0]:
# ============================================================
# Cell 1 - Configuration
# ============================================================
from pyspark.sql.functions import (monotonically_increasing_id, col, lit,
                                    when, to_date, year, month, dayofmonth,
                                    quarter, date_format)

CATALOG = "media"
SILVER  = "silver_tmdb"
GOLD    = "gold_tmdb"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}")

print(f"Ready: {CATALOG}.{GOLD}")

In [0]:
# ============================================================
# Cell 2 - Read Silver tables
# ============================================================
silver_movies = spark.table(f"{CATALOG}.{SILVER}.silver_movies")
silver_tv     = spark.table(f"{CATALOG}.{SILVER}.silver_tv_shows")
silver_mg     = spark.table(f"{CATALOG}.{SILVER}.silver_movie_genres")
silver_tg     = spark.table(f"{CATALOG}.{SILVER}.silver_tv_genres")

print("Silver tables loaded!")

In [0]:
# ============================================================
# Cell 3 - dim_title
# ============================================================
movies_titled = silver_movies.select(
    col("movie_id").alias("source_id"),
    col("title").alias("title_name"),
    col("original_title"),
    col("overview"),
    col("original_language"),
    col("popularity"),
    col("vote_average"),
    col("vote_count"),
    col("adult"),
    col("release_date").alias("air_date"),
    lit("movie").alias("media_type")
)

tv_titled = silver_tv.select(
    col("show_id").alias("source_id"),
    col("title").alias("title_name"),
    col("original_title"),
    col("overview"),
    col("original_language"),
    col("popularity"),
    col("vote_average"),
    col("vote_count"),
    col("adult"),
    col("first_air_date").alias("air_date"),
    lit("tv").alias("media_type")
)

dim_title = (movies_titled.unionByName(tv_titled)
    .dropDuplicates(["source_id", "media_type"])
    .withColumn("title_key", monotonically_increasing_id()))

(dim_title.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.dim_title"))

print(f"dim_title: {dim_title.count():,} rows")

In [0]:
# ============================================================
# Cell 4 - dim_genre
# ============================================================
dim_genre = (silver_mg.select(col("genre_id"), col("genre_name"))
    .unionByName(silver_tg.select(col("genre_id"), col("genre_name")))
    .dropDuplicates(["genre_id"])
    .withColumn("genre_key", monotonically_increasing_id()))

(dim_genre.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.dim_genre"))

print(f"dim_genre: {dim_genre.count():,} rows")

In [0]:
# ============================================================
# Cell 5 - dim_date
# ============================================================
dim_title_loaded = spark.table(f"{CATALOG}.{GOLD}.dim_title")

dim_date = (dim_title_loaded
    .select("air_date")
    .filter(col("air_date").isNotNull())
    .dropDuplicates(["air_date"])
    .withColumn("date_key",   monotonically_increasing_id())
    .withColumn("year",       year(col("air_date")))
    .withColumn("quarter",    quarter(col("air_date")))
    .withColumn("month",      month(col("air_date")))
    .withColumn("day",        dayofmonth(col("air_date")))
    .withColumn("month_name", date_format(col("air_date"), "MMMM"))
    .withColumn("is_recent",  col("year") >= 2020))

(dim_date.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.dim_date"))

print(f"dim_date: {dim_date.count():,} rows")

In [0]:
# ============================================================
# Cell 6 - dim_language
# ============================================================
dim_language = (dim_title_loaded
    .select("original_language")
    .dropDuplicates(["original_language"])
    .withColumn("language_key", monotonically_increasing_id())
    .withColumn("is_english",   col("original_language") == "en"))

(dim_language.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.dim_language"))

print(f"dim_language: {dim_language.count():,} rows")

In [0]:
# ============================================================
# Cell 7 - dim_platform
# ============================================================
dim_platform = spark.createDataFrame([
    (1, "Netflix",   "NFLX", "US"),
    (2, "Disney+",   "DIS",  "US"),
    (3, "Amazon",    "AMZN", "US"),
    (4, "Apple TV+", "AAPL", "US"),
], ["platform_key", "platform_name", "ticker", "headquarters"])

(dim_platform.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.dim_platform"))

print(f"dim_platform: {dim_platform.count():,} rows")

In [0]:
# ============================================================
# Cell 8 - fact_title_performance
# ============================================================
dim_title    = spark.table(f"{CATALOG}.{GOLD}.dim_title")
dim_genre    = spark.table(f"{CATALOG}.{GOLD}.dim_genre")
dim_date     = spark.table(f"{CATALOG}.{GOLD}.dim_date")
dim_language = spark.table(f"{CATALOG}.{GOLD}.dim_language")

all_genres = (
    silver_mg.select(col("movie_id").alias("source_id"), col("genre_id"))
    .unionByName(
        silver_tg.select(col("show_id").alias("source_id"), col("genre_id"))
    )
)

fact = (dim_title
    .join(all_genres,   dim_title.source_id == all_genres.source_id,         "left")
    .join(dim_genre,    all_genres.genre_id == dim_genre.genre_id,           "left")
    .join(dim_date,     dim_title.air_date == dim_date.air_date,             "left")
    .join(dim_language, dim_title.original_language == dim_language.original_language, "left")
    .select(
        monotonically_increasing_id().alias("fact_key"),
        dim_title.title_key,
        dim_genre.genre_key,
        dim_date.date_key,
        dim_language.language_key,
        dim_title.source_id,
        dim_title.media_type,
        dim_title.popularity,
        dim_title.vote_average,
        dim_title.vote_count,
    )
)

(fact.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD}.fact_title_performance"))

print(f"fact_title_performance: {fact.count():,} rows")

In [0]:
# ============================================================
# Cell 9 - Sanity check
# ============================================================
print("=== Gold Layer Summary ===")
for table in ["dim_title", "dim_genre", "dim_date", "dim_language",
              "dim_platform", "fact_title_performance"]:
    count = spark.table(f"{CATALOG}.{GOLD}.{table}").count()
    print(f"  {CATALOG}.{GOLD}.{table}: {count:,} rows")